# 02 — SAC/TD3 with replay
Set `AGENT` to `sac_v2` or `td3_v2` in the last cell. Random actions during warmup and epsilon-random exploration are stored in replay. Replay is not serialized: do not assume exact training resume.

In [ ]:
from pathlib import Path
import subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/MilkySon/RL_carracing_competition.git'
BRANCH = 'experiments/continuous-v2-1'
REPO = Path('/content/RL_carracing_competition')
if not (REPO / '.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
BASE = REPO/'experiments'/'continuous_ablation'/'baseline'
V2 = REPO/'experiments'/'continuous_v2_1'
ROOT = Path('/content/drive/MyDrive/CarRacing_Continuous_v2_1')
ROOT.mkdir(parents=True,exist_ok=True)
assert (BASE/'train.py').exists(), f'Baseline missing: {BASE}'
assert (V2/'scripts'/'train_v2.py').exists(), 'Upload v2.1 folder to GitHub branch first.'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(BASE/'requirements.txt')],check=True)
subprocess.run([sys.executable,str(V2/'scripts'/'install_overlay.py'),'--baseline',str(BASE)],check=True)
print('GitHub code:',V2,'Drive outputs:',ROOT)


In [ ]:
subprocess.run([sys.executable,str(V2/'scripts'/'test_boundaries.py')],check=True)
sys.path.insert(0,str(V2/'scripts'))
from test_boundaries import verify_implementation
verify_implementation(BASE)
probe_path=ROOT/'diagnostics'/'forward_probe.csv'
subprocess.run([sys.executable,str(V2/'scripts'/'forward_probe.py'),
                '--baseline',str(BASE),'--out',str(probe_path),'--seed','1000','--steps','100'],check=True)
import pandas as pd
probe=pd.read_csv(probe_path)
display(probe)
forward=float(probe.loc[probe.scenario=='forward','mean_speed'].iloc[0])
coast=float(probe.loc[probe.scenario=='coast','mean_speed'].iloc[0])
assert forward > coast + 0.1, 'FORWARD GATE FAILED: investigate environment and action mapping before training'
print('PASS: forward action increases speed compared with coasting')

In [ ]:
AGENT='sac_v2' # choose e1_v2, e2_v2, e3_v2, sac_v2, td3_v2
SEED=0
BUDGET=300000
EVAL_EVERY=50000
SHAPING=False
RUN=ROOT/AGENT/f'seed_{SEED}_'+('shaped' if SHAPING else 'raw')
# Fail rather than silently overwrite prior experiment
assert not any((RUN/x).exists() for x in ['train.csv','eval.csv','latest.pt']), f'Existing run: {RUN}'
cmd=[sys.executable,str(V2/'scripts'/'train_v2.py'),'--baseline',str(BASE),
     '--agent',AGENT,'--seed',str(SEED),'--budget',str(BUDGET),
     '--eval-every',str(EVAL_EVERY),'--run-dir',str(RUN)]
if SHAPING:cmd.append('--shaping')
print('Running:', ' '.join(cmd))
subprocess.run(cmd,check=True)